## DEVELOPMENT OF THE DEMOGRAPHICS DATASET ##

In [1]:
from pathlib import Path
from functools import reduce
import pandas as pd
import re

base = Path(r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/demographic")

files = [
    "ba_allrace_allgen.csv",
    "ba_allrace_female.csv",
    "ba_allrace_male.csv",
    "ba_black_allgen.csv",
    "ba_white_allgen.csv",
    "inc_medhousehold.csv",
    "pop_18to39.csv",
    "pop_40up.csv",
    "pop_black.csv",
    "pop_foreignborn.csv",
    "pop_white.csv",
    "pov_personsbelow.csv",
    "unemployed.csv",
]

def clean_name(x):
    x = str(x).strip().lower()
    x = re.sub(r"[^a-z0-9]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

dfs = []
for file in files:
    path = base / file
    df = pd.read_csv(path)

    # standardize column names
    df.columns = [clean_name(c) for c in df.columns]

    # standardize key columns
    if "county" not in df.columns or "fips" not in df.columns:
        raise ValueError(f"{file} must contain county and fips columns")

    df["county"] = df["county"].astype(str).str.strip()
    df["fips"] = df["fips"].astype(str).str.strip().str.zfill(5)

    # prefix non-key columns with file stem so variable names stay unique
    stem = clean_name(path.stem)
    rename_map = {}
    for c in df.columns:
        if c not in ["county", "fips"]:
            rename_map[c] = f"{stem}_{c}"
    df = df.rename(columns=rename_map)

    dfs.append(df)

demographics = reduce(
    lambda left, right: pd.merge(left, right, on=["county", "fips"], how="outer"),
    dfs
)

demographics = demographics.drop_duplicates(subset=["county", "fips"]).sort_values(["county", "fips"])

out_file = base / "demographics_merged.csv"
demographics.to_csv(out_file, index=False)

print(f"Saved: {out_file}")
print(f"Rows: {demographics.shape[0]}")
print(f"Columns: {demographics.shape[1]}")
print(demographics.columns.tolist())

Saved: C:\Users\mutch\Democratic_Voter_Shift\data\raw\demographic\demographics_merged.csv
Rows: 67
Columns: 28
['county', 'fips', 'ba_allrace_allgen_percentbaall', 'ba_allrace_allgen_edbachall', 'ba_allrace_female_percentbafemale', 'ba_allrace_female_edbachfemale', 'ba_allrace_male_percentbamale', 'ba_allrace_male_edbachmale', 'ba_black_allgen_percentbablack', 'ba_black_allgen_edbachblack', 'ba_white_allgen_percentbawhite', 'ba_white_allgen_edbachwhite', 'inc_medhousehold_dollars', 'inc_medhousehold_unnamed_3', 'pop_18to39_percent18to39', 'pop_18to39_age18to39', 'pop_40up_percent40up', 'pop_40up_age40up', 'pop_black_percentblack', 'pop_black_popblack', 'pop_foreignborn_percentforeign', 'pop_foreignborn_popforeign', 'pop_white_percentwhite', 'pop_white_popwhite', 'pov_personsbelow_percentpov', 'pov_personsbelow_poverty', 'unemployed_percentunemp', 'unemployed_unemployed']


In [2]:
df = pd.read_csv(r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/demographic/Demographics_merged.csv")

df["county"] = (
    df["county"]
    .astype(str)
    .str.replace(r"\s+County\s*$", "", regex=True)
    .str.strip()
)

df.to_csv(
    r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/demographic/Demographics_merged.csv",
    index=False
)

print("County column cleaned and file saved.")

County column cleaned and file saved.


In [3]:
import pandas as pd
from pathlib import Path

fp = Path(r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/demographic/Demographics_merged.csv")
df = pd.read_csv(fp)

# Drop the column if it exists
if "inc_medhousehold_unnamed_3" in df.columns:
    df = df.drop(columns=["inc_medhousehold_unnamed_3"])
    df.to_csv(fp, index=False)
    print("Dropped column inc_medhousehold_unnamed_3 and saved file.")
else:
    print("Column inc_medhousehold_unnamed_3 not found; no changes made.")

Dropped column inc_medhousehold_unnamed_3 and saved file.


In [4]:
col_info = pd.DataFrame({
    "column_name": df.columns,
    "data_type": df.dtypes.astype(str).values
})

print(col_info)

                          column_name data_type
0                              county       str
1                                fips     int64
2      ba_allrace_allgen_percentbaall   float64
3         ba_allrace_allgen_edbachall     int64
4   ba_allrace_female_percentbafemale   float64
5      ba_allrace_female_edbachfemale     int64
6       ba_allrace_male_percentbamale   float64
7          ba_allrace_male_edbachmale     int64
8      ba_black_allgen_percentbablack   float64
9         ba_black_allgen_edbachblack     int64
10     ba_white_allgen_percentbawhite   float64
11        ba_white_allgen_edbachwhite     int64
12           inc_medhousehold_dollars       str
13           pop_18to39_percent18to39   float64
14               pop_18to39_age18to39     int64
15               pop_40up_percent40up   float64
16                   pop_40up_age40up     int64
17             pop_black_percentblack   float64
18                 pop_black_popblack     int64
19     pop_foreignborn_percentforeign   

## RENAMING THE DEMOGRAPHIC VARIABLES ##

In [1]:
import pandas as pd
from pathlib import Path

# Set file paths
input_file = Path("C:/Users/mutch/Democratic_Voter_Shift/data/raw/demographic/demographics_merged.csv")
output_file = Path("C:/Users/mutch/Democratic_Voter_Shift/data/raw/demographic/demographic_merged_renamed.csv")

# Read the merged dataset
df = pd.read_csv(input_file)

# Define the new column names in exact order
new_columns = [
    "county", "fips",
    "pct_ba", "num_ba",
    "pct_bafemale", "num_bafemale",
    "pct_bamale", "num_bamale",
    "pct_bablack", "num_bablack",
    "pct_bawhite", "num_bawhite",
    "income",
    "pct_pop18to39", "num_pop18to39",
    "pct_pop40up", "num_pop40up",
    "pct_black", "num_black",
    "pct_foreignborn", "num_foreignborn",
    "pct_white", "num_white",
    "pct_poverty", "num_poverty",
    "pct_unemployed", "num_unemployed"
]

# Check that the dataset has the expected number of columns
if len(df.columns) != len(new_columns):
    raise ValueError(
        f"Column count mismatch: dataset has {len(df.columns)} columns, "
        f"but {len(new_columns)} names were provided."
    )

# Rename columns
df.columns = new_columns

# Save renamed dataset
df.to_csv(output_file, index=False)

# Preview the result
df.head()

,county,fips,pct_ba,num_ba,pct_bafemale,num_bafemale,pct_bamale,num_bamale,pct_bablack,num_bablack,...,pct_black,num_black,pct_foreignborn,num_foreignborn,pct_white,num_white,pct_poverty,num_poverty,pct_unemployed,num_unemployed
0,Adams,42001,25.8,19282,28.5,10904,22.9,8378,22.9,271,...,1.8,1952,3.7,3942,88.9,94377,8.2,8336,3.4,1885
1,Allegheny,42003,45.8,411770,45.2,211471,46.3,200299,22.9,22740,...,12.5,154273,6.6,81618,76.1,941697,11.5,138617,5.0,33744
2,Armstrong,42005,19.0,9154,21.0,5088,17.0,4066,0.2,1,...,0.9,608,0.7,482,95.2,61501,11.8,7543,4.3,1335
3,Beaver,42007,28.7,35127,29.3,18569,28.1,16558,17.0,1131,...,6.2,10367,1.9,3080,86.9,144595,9.9,16198,5.3,4498
4,Bedford,42009,16.1,5627,18.3,3241,13.8,2386,13.4,22,...,0.7,342,1.2,582,95.5,45397,11.6,5429,3.9,885


**adding population of under 18 to merged file**

In [3]:
import pandas as pd
from pathlib import Path

# File paths
demographic_file = Path(r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/demographic/demographic_merged_renamed.csv")
pop_below18_file = Path(r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/demographic/pop_below18.csv")

# Read the files
df_demo = pd.read_csv(demographic_file)
df_pop = pd.read_csv(pop_below18_file)

# Optional: standardize key text columns
for df in [df_demo, df_pop]:
    if "county" in df.columns:
        df["county"] = df["county"].astype(str).str.strip().str.lower()

# Merge on county or fips, depending on what the files have in common
if "fips" in df_demo.columns and "fips" in df_pop.columns:
    merged = df_demo.merge(df_pop, on="fips", how="left")
elif "county" in df_demo.columns and "county" in df_pop.columns:
    merged = df_demo.merge(df_pop, on="county", how="left")
else:
    raise ValueError("No common merge key found. Check whether both files share 'county' or 'fips'.")

# Save the merged file
output_file = Path(r"C:/Users/mutch/Democratic_Voter_Shift/data/raw/demographic_all.csv")
merged.to_csv(output_file, index=False)

# Preview
merged.head()

,county_x,fips,pct_ba,num_ba,pct_bafemale,num_bafemale,pct_bamale,num_bamale,pct_bablack,num_bablack,...,pct_white,num_white,pct_poverty,num_poverty,pct_unemployed,num_unemployed,county_y,pct_below18,num_below18,Unnamed: 4
0,adams,42001,25.8,19282,28.5,10904,22.9,8378,22.9,271,...,88.9,94377,8.2,8336,3.4,1885,adams,19.4,20638,NaN
1,allegheny,42003,45.8,411770,45.2,211471,46.3,200299,22.9,22740,...,76.1,941697,11.5,138617,5.0,33744,allegheny,18.6,230228,NaN
2,armstrong,42005,19.0,9154,21.0,5088,17.0,4066,0.2,1,...,95.2,61501,11.8,7543,4.3,1335,armstrong,18.9,12227,NaN
3,beaver,42007,28.7,35127,29.3,18569,28.1,16558,17.0,1131,...,86.9,144595,9.9,16198,5.3,4498,beaver,19.4,32226,NaN
4,bedford,42009,16.1,5627,18.3,3241,13.8,2386,13.4,22,...,95.5,45397,11.6,5429,3.9,885,bedford,19.4,9234,NaN
